# Graph Complexity Benchmark Results

This notebook loads and displays performance metrics (	ime_of_parsing, 	ime_of_rendering, 	ime_for_metric_computation, memory_peak_mb) for the **Planetoid** and **MovieLens 100K** datasets across different node configurations.

In [1]:
import json
from pathlib import Path
import pandas as pd

In [2]:
def load_metrics_df(dataset_name: str) -> pd.DataFrame:
    """Load all metric JSON files for a dataset and return as a DataFrame."""
    metrics_dirs = [
        Path(f"checkpoints/{dataset_name}/metrics"),
        Path(f"checkpoints/{dataset_name}/metrics"),
        Path(f"checkpoints/{dataset_name}"),
        Path(f"checkpoints/{dataset_name}"),
        Path(f"data/metrics/results/{dataset_name}"),
        Path(f"data/metrics/results/{dataset_name}"),
    ]
    
    if "movielens" in dataset_name.lower():
        metrics_dirs.extend([
            Path("checkpoints/movielens100k/metrics"),
            Path("checkpoints/movielens100k/metrics"),
            Path("checkpoints/movielens/metrics"),
            Path("checkpoints/movielens/metrics"),
        ])
        
    target_dir = None
    for d in metrics_dirs:
        if d.exists() and list(d.glob("*.json")):
            target_dir = d
            break
            
    if target_dir is None:
        print(f"Warning: No metrics JSON directory found for {dataset_name}")
        return pd.DataFrame()

    rows = []
    for json_file in sorted(target_dir.glob("*.json"), key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem):
        if json_file.stem == "summary":
            continue
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            rows.append(data)
        except Exception as e:
            print(f"Error reading {json_file}: {e}")

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    # Sort numerically by number_of_nodes
    if "number_of_nodes" in df.columns:
        df = df.sort_values("number_of_nodes").reset_index(drop=True)

    # Reorder columns with metrics
    columns_order = [
        "number_of_nodes",
        "number_of_edges",
        "time_of_parsing",
        "time_of_rendering",
        "time_for_metric_computation",
        "memory_peak_mb",
    ]
    ordered_cols = [c for c in columns_order if c in df.columns] + [c for c in df.columns if c not in columns_order]
    df = df[ordered_cols]

    return df


def format_metrics_table(df: pd.DataFrame):
    """Format DataFrame with plain numbers for node/edge counts and .2f for timing/memory metrics."""
    if df.empty:
        return df

    df_formatted = df.copy()
    
    # Cast node/edge counts to int
    for col in ["number_of_nodes", "number_of_edges"]:
        if col in df_formatted.columns:
            df_formatted[col] = df_formatted[col].fillna(0).astype(int)

    format_dict = {
        "number_of_nodes": "{:d}",
        "number_of_edges": "{:d}",
        "time_of_parsing": "{:.2f}",
        "time_of_rendering": "{:.2f}",
        "time_for_metric_computation": "{:.2f}",
        "memory_peak_mb": "{:.2f}",
    }
    
    applied_format = {k: v for k, v in format_dict.items() if k in df_formatted.columns}
    
    return df_formatted.style.format(applied_format)

## 1. Planetoid Dataset Complexity Metrics

In [3]:
planetoid_df = load_metrics_df("planetoid")
planetoid_df = format_metrics_table(planetoid_df)
planetoid_df

,number_of_nodes,number_of_edges,time_of_parsing,time_of_rendering,time_for_metric_computation,memory_peak_mb
0,100,197,0.03,0.18,0.00,3.58
1,500,1067,0.26,0.76,0.00,17.72
2,1000,2098,0.79,1.56,0.00,36.13
3,2485,5069,3.76,4.57,0.00,91.17
4,2708,5278,4.17,4.96,0.00,95.95
5,2708,5278,4.19,4.92,0.00,95.96


In [4]:
print(planetoid_df.to_latex())

\begin{tabular}{lrrrrrr}
 & number_of_nodes & number_of_edges & time_of_parsing & time_of_rendering & time_for_metric_computation & memory_peak_mb \\
0 & 100 & 197 & 0.03 & 0.18 & 0.00 & 3.58 \\
1 & 500 & 1067 & 0.26 & 0.76 & 0.00 & 17.72 \\
2 & 1000 & 2098 & 0.79 & 1.56 & 0.00 & 36.13 \\
3 & 2485 & 5069 & 3.76 & 4.57 & 0.00 & 91.17 \\
4 & 2708 & 5278 & 4.17 & 4.96 & 0.00 & 95.95 \\
5 & 2708 & 5278 & 4.19 & 4.92 & 0.00 & 95.96 \\
\end{tabular}



## 2. MovieLens 100K Dataset Complexity Metrics

In [5]:
movielens_df = load_metrics_df("movielens")
movielens_df = format_metrics_table(movielens_df)
movielens_df

,number_of_nodes,number_of_edges,time_of_parsing,time_of_rendering,time_for_metric_computation,memory_peak_mb
0,88,440,0.07,0.27,0.00,5.74
1,476,10378,8.16,6.54,0.01,105.87
2,999,48182,146.30,184.41,0.02,457.79
3,2185,156692,1583.22,2366.31,0.12,1541.98
4,2593,160000,1589.34,2517.71,0.22,1570.76


In [6]:
print(movielens_df.to_latex())

\begin{tabular}{lrrrrrr}
 & number_of_nodes & number_of_edges & time_of_parsing & time_of_rendering & time_for_metric_computation & memory_peak_mb \\
0 & 88 & 440 & 0.07 & 0.27 & 0.00 & 5.74 \\
1 & 476 & 10378 & 8.16 & 6.54 & 0.01 & 105.87 \\
2 & 999 & 48182 & 146.30 & 184.41 & 0.02 & 457.79 \\
3 & 2185 & 156692 & 1583.22 & 2366.31 & 0.12 & 1541.98 \\
4 & 2593 & 160000 & 1589.34 & 2517.71 & 0.22 & 1570.76 \\
\end{tabular}

